# PoC học thuật: phát hiện giao dịch nghi ngờ gian lận

Chỉ dùng dữ liệu tổng hợp. Không đại diện MB, không kết nối ngân hàng và không chặn giao dịch thật.


In [ ]:
from pathlib import Path
FILES = {"train_model.py": "\"\"\"Reproducible academic PoC; all records and labels are synthetic, not MB data.\"\"\"\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport platform\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport sklearn\nimport xgboost as xgb\nfrom sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,\n                             fbeta_score, precision_recall_curve, precision_score,\n                             recall_score, roc_auc_score)\nfrom sklearn.model_selection import train_test_split\n\nFEATURES = ['amount', 'hour', 'channel', 'new_device', 'location_anomaly',\n            'failed_attempts', 'tx_frequency_10m', 'new_beneficiary']\nBASE = Path(__file__).resolve().parent\nSCHEMA_VERSION = '1.1'\nSEEDS = (42, 52, 62)\n\n\ndef generate_synthetic_bank_data(n_samples: int = 25_000, random_state: int = 42) -> pd.DataFrame:\n    \"\"\"Generate synthetic transactions; the 1.8% label rate is a design choice.\"\"\"\n    if not isinstance(n_samples, int) or n_samples < 100:\n        raise ValueError('n_samples phải là số nguyên >= 100.')\n    rng = np.random.default_rng(random_state)\n    amount = np.clip(rng.lognormal(mean=13.5, sigma=1.5, size=n_samples), 20_000, 500_000_000)\n    hour_weights = np.array([.01,.005,.005,.005,.01,.02,.04,.06,.08,.08,.07,.06,\n                             .05,.06,.08,.08,.07,.06,.06,.05,.03,.02,.015,.01])\n    hour = rng.choice(np.arange(24), size=n_samples, p=hour_weights/hour_weights.sum())\n    channel = rng.choice([0,1,2,3], size=n_samples, p=[.70,.15,.10,.05])\n    new_device = rng.choice([0,1], size=n_samples, p=[.92,.08])\n    location_anomaly = rng.choice([0,1], size=n_samples, p=[.95,.05])\n    failed_attempts = rng.choice([0,1,2,3], size=n_samples, p=[.88,.08,.03,.01])\n    tx_frequency_10m = np.clip(rng.geometric(p=.4, size=n_samples),1,20)\n    new_beneficiary = rng.choice([0,1], size=n_samples, p=[.15,.85])\n    latent = ((amount > 30_000_000).astype(int)*2.0 +\n              (amount > 100_000_000).astype(int)*3.0 +\n              ((hour >= 1)&(hour <= 4)).astype(int)*2.5 + new_device*3.5 +\n              location_anomaly*4.0 + (failed_attempts >= 2).astype(int)*3.0 +\n              (tx_frequency_10m >= 5).astype(int)*2.5 + (new_beneficiary == 0).astype(int)*2.0)\n    latent += rng.normal(0,1.2,size=n_samples)\n    label_cut = np.percentile(latent,98.2)\n    is_fraud = (latent >= label_cut).astype(int)\n    return pd.DataFrame({\n        'amount':amount, 'hour':hour, 'channel':channel, 'new_device':new_device,\n        'location_anomaly':location_anomaly, 'failed_attempts':failed_attempts,\n        'tx_frequency_10m':tx_frequency_10m, 'new_beneficiary':new_beneficiary,\n        'is_fraud':is_fraud\n    })\n\n\ndef rule_score(X: pd.DataFrame) -> np.ndarray:\n    \"\"\"Transparent rule baseline on three features, distinct from label formula.\"\"\"\n    return ((X.new_device == 1).astype(float) +\n            (X.location_anomaly == 1).astype(float) +\n            (X.amount > 30_000_000).astype(float)).to_numpy()/3.0\n\n\ndef choose_f2_threshold(y_true, scores):\n    \"\"\"Maximize validation F2; ties choose the smaller validation alert rate.\"\"\"\n    y = np.asarray(y_true, dtype=int)\n    s = np.asarray(scores, dtype=float)\n    thresholds = np.unique(s)\n    thresholds = np.append(thresholds, np.nextafter(thresholds.max(), np.inf))\n    candidates = []\n    for threshold in thresholds:\n        pred = s >= threshold\n        candidates.append((float(fbeta_score(y,pred,beta=2,zero_division=0)),\n                           float(pred.mean()), float(threshold)))\n    best_f2 = max(r[0] for r in candidates)\n    tied = [r for r in candidates if np.isclose(r[0],best_f2,rtol=0,atol=1e-12)]\n    return min(tied,key=lambda r:(r[1],-r[2]))\n\n\ndef evaluate(y_true, scores, threshold):\n    y = np.asarray(y_true,dtype=int); s = np.asarray(scores,dtype=float)\n    pred = (s >= threshold).astype(int)\n    tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()\n    return {\n        'roc_auc':float(roc_auc_score(y,s)),\n        'average_precision':float(average_precision_score(y,s)),\n        'precision':float(precision_score(y,pred,zero_division=0)),\n        'recall':float(recall_score(y,pred,zero_division=0)),\n        'f1':float(f1_score(y,pred,zero_division=0)),\n        'f2':float(fbeta_score(y,pred,beta=2,zero_division=0)),\n        'false_positive_rate':float(fp/(fp+tn)) if fp+tn else 0.0,\n        'alert_rate':float(pred.mean()),\n        'confusion_matrix':[[int(tn),int(fp)],[int(fn),int(tp)]]\n    }\n\n\ndef _threshold_tradeoff(y_true, scores, model_name):\n    y=np.asarray(y_true,dtype=int); s=np.asarray(scores,dtype=float)\n    unique=np.unique(s)\n    if len(unique)>1200:\n        idx=np.unique(np.linspace(0,len(unique)-1,1200).round().astype(int))\n        unique=unique[idx]\n    values=[]\n    for t in unique:\n        m=evaluate(y,s,float(t))\n        values.append({'model':model_name,'threshold':float(t),'precision':m['precision'],\n            'recall':m['recall'],'f2':m['f2'],'false_positive_rate':m['false_positive_rate'],\n            'alert_rate':m['alert_rate']})\n    return values\n\n\ndef _measure_inference_latency(model, X_test, repetitions=250):\n    row=X_test.iloc[[0]]\n    for _ in range(30): model.predict_proba(row)\n    times=[]\n    for i in range(repetitions):\n        sample=X_test.iloc[[i % len(X_test)]]\n        start=time.perf_counter_ns(); model.predict_proba(sample); times.append((time.perf_counter_ns()-start)/1e6)\n    return {'method':'single-row predict_proba; 30 warm-ups, 250 measured calls; local process only',\n            'median_ms':float(np.median(times)),'p95_ms':float(np.percentile(times,95)),\n            'n_warmup':30,'n_measured':repetitions}\n\n\ndef train_and_save(output_dir=BASE, n_samples=25_000):\n    out=Path(output_dir); out.mkdir(parents=True,exist_ok=True)\n    df=generate_synthetic_bank_data(n_samples=n_samples,random_state=42)\n    X=df[FEATURES].reset_index(drop=True); y=df.is_fraud.reset_index(drop=True)\n    X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)\n    X_train,X_val,y_train,y_val=train_test_split(X_dev,y_dev,test_size=.25,random_state=42,stratify=y_dev)\n    # Fixed data/splits across seeds; seed variation measures training stochasticity only.\n    data_digest=hashlib.sha256(pd.concat([X,y.rename('is_fraud')],axis=1).to_csv(index=False).encode()).hexdigest()\n    rule_val=rule_score(X_val); rule_pick=choose_f2_threshold(y_val,rule_val)\n    rule_test=rule_score(X_test)\n    baselines={\n        'always_negative':evaluate(y_test,np.zeros(len(y_test)),.5),\n        'simple_rule':evaluate(y_test,rule_test,rule_pick[2])\n    }\n    baselines['simple_rule']['validation_threshold']=rule_pick[2]\n    results={}; models={}; predictions={\n        'synthetic_row_id':[f'SYN-{i:05d}' for i in X_test.index],\n        'synthetic_label':y_test.to_numpy(),\n        'rule_score':rule_test,\n        'rule_pred':(rule_test>=rule_pick[2]).astype(int),\n        'always_negative_pred':np.zeros(len(y_test),dtype=int)\n    }\n    seed_rows=[]; tradeoff=[]; pr_rows=[]\n    for seed in SEEDS:\n        model=xgb.XGBClassifier(n_estimators=150,max_depth=5,learning_rate=.08,\n            scale_pos_weight=float((y_train==0).sum()/y_train.sum()),subsample=.8,\n            colsample_bytree=.8,eval_metric='aucpr',random_state=seed,n_jobs=2)\n        model.fit(X_train,y_train)\n        val_scores=model.predict_proba(X_val)[:,1]\n        threshold_pick=choose_f2_threshold(y_val,val_scores)\n        test_scores=model.predict_proba(X_test)[:,1]\n        item=evaluate(y_test,test_scores,threshold_pick[2])\n        item.update({'threshold':threshold_pick[2], 'validation_f2':threshold_pick[0],\n                     'validation_alert_rate':threshold_pick[1]})\n        results[str(seed)]=item; models[seed]=model\n        predictions[f'model_score_seed_{seed}']=test_scores\n        predictions[f'model_pred_seed_{seed}']=(test_scores>=threshold_pick[2]).astype(int)\n        seed_rows.append({'seed':seed,'threshold':threshold_pick[2],**{k:v for k,v in item.items() if k!='confusion_matrix'}})\n        if seed==42:\n            tradeoff.extend(_threshold_tradeoff(y_test,test_scores,'XGBoost seed 42'))\n            p,r,t=precision_recall_curve(y_test,test_scores)\n            ap=float(average_precision_score(y_test,test_scores))\n            pr_rows.extend([{'model':'XGBoost seed 42','recall':float(r[i]),'precision':float(p[i]),\n                            'threshold':float(t[i]) if i<len(t) else None,'average_precision':ap}\n                           for i in range(len(p))])\n    tradeoff.extend(_threshold_tradeoff(y_test,rule_test,'Simple rules'))\n    model_path=out/'fraud_model.json'; models[42].save_model(model_path)\n    model_sha=hashlib.sha256(model_path.read_bytes()).hexdigest()\n    summary={}\n    for metric in ['roc_auc','average_precision','precision','recall','f1','f2','false_positive_rate','alert_rate']:\n        values=np.array([results[str(s)][metric] for s in SEEDS],dtype=float)\n        summary[metric]={'mean':float(values.mean()),'sample_sd':float(values.std(ddof=1))}\n    report={\n        'schema_version':SCHEMA_VERSION,'run_id':f\"SYN25K-S42-SPLIT42-MODEL42_52_62-{data_digest[:12]}\",\n        'data_source':'synthetic_only_not_MB','data_sha256':data_digest,\n        'label_generation':'synthetic latent risk score + Gaussian noise; top 1.8 percentile labeled positive',\n        'n_samples':len(df),'positive_rate':float(y.mean()),\n        'split':{'train':len(y_train),'validation':len(y_val),'test':len(y_test),\n                 'ratios':'60/20/20 stratified','same_split_for_all_model_seeds':True},\n        'selection_protocol':{'metric':'F2','beta':2,'selection_data':'validation only',\n           'candidate_thresholds':'unique validation scores plus nextafter(max, +inf)',\n           'tie_break':'lower validation alert rate, then higher threshold',\n           'test_used_for_threshold_selection':False},\n        'demo_policy':{'model_seed':42,'high_threshold':max(.75,float(results['42']['threshold'])),\n            'high_threshold_status':'display-only policy assumption; fixed minimum 0.75, not tuned or approved by MB'},\n        'model_sha256':model_sha,\n        'feature_order':FEATURES,\n        'metrics':{'always_negative':baselines['always_negative'],'simple_rule':baselines['simple_rule'],\n                   'xgboost_by_seed':results,'xgboost_seed_summary':summary},\n        'latency_seed_42':_measure_inference_latency(models[42],X_test),\n        'environment':{'python':platform.python_version(),'platform':platform.platform(),\n             'machine':platform.machine(),'cpu_count':__import__('os').cpu_count(),\n             'numpy':np.__version__,'pandas':pd.__version__,'scikit_learn':sklearn.__version__,\n             'xgboost':xgb.__version__},\n        'created_at_utc':datetime.now(timezone.utc).isoformat()\n    }\n    metric_path=out/'metrics.json'; metric_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')\n    pd.DataFrame(predictions).to_csv(out/'test_predictions.csv',index=False)\n    pd.DataFrame(seed_rows).to_csv(out/'seed_results.csv',index=False)\n    pd.DataFrame(tradeoff).to_csv(out/'threshold_tradeoff.csv',index=False)\n    pd.DataFrame(pr_rows).to_csv(out/'pr_curve_seed42.csv',index=False)\n    booster=models[42].get_booster()\n    importance=pd.DataFrame({'feature':FEATURES,'gain':booster.get_score(importance_type='gain').values()})\n    gain=booster.get_score(importance_type='gain')\n    importance=pd.DataFrame([{'feature':f,'gain':float(gain.get(f,0))} for f in FEATURES]).sort_values('gain',ascending=False)\n    importance.to_csv(out/'feature_importance_seed42.csv',index=False)\n    manifest={'schema_version':SCHEMA_VERSION,'run_id':report['run_id'],'model_file':model_path.name,\n      'model_sha256':model_sha,'metrics_file':metric_path.name,\n      'metrics_sha256':hashlib.sha256(metric_path.read_bytes()).hexdigest(),\n      'feature_order':FEATURES,'seed':42}\n    (out/'artifact_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')\n    print(json.dumps(report,ensure_ascii=False,indent=2))\n    return models[42],report\n\nif __name__=='__main__':\n    train_and_save()\n", "artifact_loader.py": "\"\"\"Fail-closed loading for model + evaluation artifacts.\"\"\"\nimport hashlib\nimport json\nfrom pathlib import Path\nimport xgboost as xgb\nfrom train_model import FEATURES, SCHEMA_VERSION\n\n\ndef verify_artifacts(directory):\n    base=Path(directory)\n    manifest=json.loads((base/'artifact_manifest.json').read_text(encoding='utf-8'))\n    metrics_path=base/manifest['metrics_file']; model_path=base/manifest['model_file']\n    metrics_sha=hashlib.sha256(metrics_path.read_bytes()).hexdigest()\n    model_sha=hashlib.sha256(model_path.read_bytes()).hexdigest()\n    if manifest.get('schema_version')!=SCHEMA_VERSION: raise ValueError('Sai schema_version trong manifest.')\n    if manifest.get('feature_order')!=FEATURES: raise ValueError('Thứ tự feature trong manifest không khớp.')\n    if metrics_sha!=manifest.get('metrics_sha256'): raise ValueError('metrics.json không cùng artifact manifest.')\n    if model_sha!=manifest.get('model_sha256'): raise ValueError('Model không cùng artifact manifest.')\n    metrics=json.loads(metrics_path.read_text(encoding='utf-8'))\n    if metrics.get('model_sha256')!=model_sha or metrics.get('run_id')!=manifest.get('run_id'):\n        raise ValueError('Run ID hoặc hash model/metrics không nhất quán.')\n    model=xgb.XGBClassifier(); model.load_model(model_path)\n    if model.get_booster().feature_names!=FEATURES: raise ValueError('Feature model không khớp contract.')\n    return model,metrics,manifest\n", "risk_engine.py": "\"\"\"Shared scoring and demo policy. It does not authorize or block transactions.\"\"\"\nimport math\nimport time\nfrom numbers import Real\nimport pandas as pd\nfrom train_model import FEATURES\n\nBOUNDS={'amount':(20_000,500_000_000),'hour':(0,23),'channel':(0,3),\n        'new_device':(0,1),'location_anomaly':(0,1),'failed_attempts':(0,3),\n        'tx_frequency_10m':(1,20),'new_beneficiary':(0,1)}\n\n\ndef validate(transaction):\n    if not isinstance(transaction,dict) or set(transaction)!=set(FEATURES):\n        raise ValueError('Giao dịch phải có đúng 8 trường đặc trưng.')\n    normalized={}\n    for key,(lo,hi) in BOUNDS.items():\n        value=transaction[key]\n        if isinstance(value,bool) or not isinstance(value,Real) or not math.isfinite(float(value)) or not lo<=value<=hi:\n            raise ValueError(f'{key}: yêu cầu số hữu hạn trong [{lo}, {hi}].')\n        if key!='amount' and float(value)!=int(value):\n            raise ValueError(f'{key}: phải là số nguyên.')\n        normalized[key]=float(value) if key=='amount' else int(value)\n    return pd.DataFrame([normalized],columns=FEATURES)\n\n\ndef action(score, threshold, high_threshold):\n    if not 0<=score<=1 or not 0<=threshold<=high_threshold<=1:\n        raise ValueError('Điểm hoặc ngưỡng chính sách demo không hợp lệ.')\n    if score>=high_threshold: return 'Đề xuất chuyển chuyên viên xem xét'\n    if score>=threshold: return 'Đề xuất xác thực bổ sung'\n    return 'Rủi ro thấp; giữ nguyên các kiểm soát thông thường'\n\n\ndef score_transaction(model, report, transaction):\n    frame=validate(transaction); started=time.perf_counter_ns()\n    score=float(model.predict_proba(frame)[0,1])\n    latency=(time.perf_counter_ns()-started)/1e6\n    threshold=float(report['metrics']['xgboost_by_seed']['42']['threshold'])\n    high=float(report['demo_policy']['high_threshold'])\n    return {'score':score,'latency_ms':latency,'action':action(score,threshold,high),\n            'model_seed':42,'run_id':report['run_id']}\n", "app.py": "from pathlib import Path\nimport pandas as pd\nimport streamlit as st\nimport xgboost as xgb\nfrom artifact_loader import verify_artifacts\nfrom risk_engine import score_transaction, validate\nfrom train_model import FEATURES, generate_synthetic_bank_data\n\nBASE=Path(__file__).resolve().parent\nst.set_page_config(page_title='PoC học thuật | Phát hiện giao dịch nghi ngờ',page_icon='🛡️',layout='wide')\nst.title('AI hỗ trợ phát hiện giao dịch nghi ngờ gian lận')\nst.caption('Bối cảnh nghiên cứu đề xuất: MB | Học phần Công nghệ mới nổi và Đổi mới – HVNH')\nst.warning('PoC học thuật trên dữ liệu tổng hợp, không phải hệ thống MB. Không kết nối ngân hàng, không chặn tiền, không xác thực khách hàng và không chứng nhận tuân thủ pháp lý.')\n\n@st.cache_resource\ndef load_checked_artifacts(stamp_model,stamp_metrics,stamp_manifest):\n    return verify_artifacts(BASE)\ntry:\n    model,report,manifest=load_checked_artifacts(\n        (BASE/'fraud_model.json').stat().st_mtime_ns,\n        (BASE/'metrics.json').stat().st_mtime_ns,\n        (BASE/'artifact_manifest.json').stat().st_mtime_ns)\nexcept Exception as exc:\n    st.error(f'Không thể xác minh bộ mô hình và kết quả: {exc}. Hãy chạy python train_model.py; ứng dụng dừng chấm điểm khi artifact không khớp.')\n    st.stop()\nif 'history' not in st.session_state: st.session_state.history=[]\nhistory=st.session_state.history\nthreshold=float(report['metrics']['xgboost_by_seed']['42']['threshold'])\nhigh=float(report['demo_policy']['high_threshold'])\ncols=st.columns(4)\ncols[0].metric('Giao dịch mô phỏng trong phiên',len(history))\ncols[1].metric('Cần xem xét trong phiên',sum(r['score']>=threshold for r in history))\ncols[2].metric('Ngưỡng mô hình seed 42',f'{threshold:.3f}')\nlat=[r['latency_ms'] for r in history]\ncols[3].metric('Inference trung vị (ms)',f'{pd.Series(lat).median():.2f}' if lat else 'Chưa đo')\nst.caption('Độ trễ là predict_proba cho một giao dịch trên máy hiện tại; không gồm mạng, xác thực hay xử lý giao dịch và không phải SLA.')\ninput_tab,feed_tab,eval_tab,architecture_tab=st.tabs(['Giả lập giao dịch','Lịch sử mô phỏng','Đánh giá mô hình','Kiến trúc & phạm vi'])\nwith input_tab:\n    st.info(f'Chính sách giao diện chỉ để minh họa: thấp < {threshold:.3f}; xác thực bổ sung từ {threshold:.3f} đến dưới {high:.3f}; chuyển chuyên viên từ {high:.3f}. Điểm chưa được hiệu chỉnh thành xác suất thực tế.')\n    with st.form('transaction'):\n        left,right=st.columns(2)\n        with left:\n            amount=st.number_input('Số tiền (VND)',20_000,500_000_000,1_500_000,10_000)\n            hour=st.slider('Giờ giao dịch',0,23,14)\n            channel=st.selectbox('Kênh mô phỏng',[0,1,2,3],format_func=lambda i:['Mobile','Web','Thẻ/POS','Ví điện tử'][i])\n            frequency=st.slider('Số giao dịch trong 10 phút, gồm giao dịch hiện tại',1,20,1)\n        with right:\n            device=st.checkbox('Thiết bị mới (tín hiệu mô phỏng)')\n            location=st.checkbox('Vị trí bất thường (tín hiệu mô phỏng)')\n            failed=st.slider('Số lần sai OTP/PIN trong 30 phút (3 = từ 3 trở lên)',0,3,0)\n            beneficiary=st.checkbox('Người nhận chưa từng được người gửi chuyển trước đó')\n        submit=st.form_submit_button('Chấm điểm mô phỏng',type='primary')\n    if submit:\n        tx={'amount':amount,'hour':hour,'channel':channel,'new_device':int(device),\n            'location_anomaly':int(location),'failed_attempts':failed,\n            'tx_frequency_10m':frequency,'new_beneficiary':0 if beneficiary else 1}\n        try:\n            result=score_transaction(model,report,tx)\n            st.session_state.history.append({'demo_id':f\"DEMO-{len(history)+1:05d}\",**tx,**result})\n            st.session_state.last_tx=tx; st.session_state.last_result=result; st.rerun()\n        except Exception as exc:\n            st.error(f'Không thể chấm điểm: {exc}. Chuyển kiểm tra thủ công; không tự phê duyệt.')\n    if 'last_result' in st.session_state:\n        result=st.session_state.last_result\n        st.subheader('Kết quả chấm điểm')\n        st.metric('Điểm mô hình (0–100)',f\"{result['score']*100:.2f}\")\n        st.write('**Hành động đề xuất:**',result['action'])\n        st.caption(f\"Inference đo trên máy hiện tại: {result['latency_ms']:.3f} ms | Model seed: {result['model_seed']} | Run ID: {result['run_id']}\")\n        frame=validate(st.session_state.last_tx)\n        contrib=model.get_booster().predict(xgb.DMatrix(frame),pred_contribs=True)[0]\n        imp=pd.DataFrame({'Đóng góp raw margin':contrib[:-1]},index=FEATURES).sort_values('Đóng góp raw margin')\n        st.subheader('Đóng góp TreeSHAP cho giao dịch mô phỏng')\n        st.bar_chart(imp)\n        st.caption(f'Bias={contrib[-1]:.4f}; tổng đóng góp={contrib.sum():.4f} ở raw margin. Giá trị này giải thích đầu ra của mô hình, không chứng minh nguyên nhân hay gian lận đã xảy ra.')\nwith feed_tab:\n    st.caption('Batch tổng hợp, không phải streaming hoặc dữ liệu giao dịch trực tiếp. Lịch sử chỉ tồn tại trong phiên trình duyệt.')\n    b1,b2=st.columns(2)\n    with b1:\n        if st.button('Sinh 20 giao dịch tổng hợp'):\n            batch=generate_synthetic_bank_data(20,random_state=42+len(history))\n            for _,r in batch[FEATURES].iterrows():\n                tx={k:float(r[k]) if k=='amount' else int(r[k]) for k in FEATURES}\n                st.session_state.history.append({'demo_id':f\"DEMO-{len(st.session_state.history)+1:05d}\",**tx,**score_transaction(model,report,tx)})\n            st.rerun()\n    with b2:\n        if st.button('Xóa lịch sử phiên'):\n            st.session_state.history=[]; st.session_state.pop('last_result',None); st.session_state.pop('last_tx',None); st.rerun()\n    if history:\n        table=pd.DataFrame(history)\n        st.dataframe(table,use_container_width=True,hide_index=True)\n        st.download_button('Tải CSV lịch sử mô phỏng',table.to_csv(index=False).encode('utf-8-sig'),\n                           file_name='demo_history.csv',mime='text/csv')\n    else: st.info('Chưa có giao dịch trong phiên.')\nwith eval_tab:\n    st.caption(f\"Run: {report['run_id']} | Split: train {report['split']['train']:,}, validation {report['split']['validation']:,}, test {report['split']['test']:,} | Nhãn dương tổng hợp: {report['positive_rate']:.2%}\")\n    metrics=report['metrics']; seed42=metrics['xgboost_by_seed']['42']\n    comparison=pd.DataFrame({\n        'Luôn dự đoán bình thường':metrics['always_negative'],\n        'Quy tắc đơn giản':metrics['simple_rule'],\n        'XGBoost seed 42':{k:v for k,v in seed42.items() if k!='confusion_matrix'}\n    }).T\n    st.subheader('So sánh trên tập test holdout')\n    st.dataframe(comparison,use_container_width=True)\n    st.write('Confusion matrix XGBoost seed 42: hàng = nhãn thật [0,1], cột = dự đoán [0,1].')\n    st.dataframe(pd.DataFrame(seed42['confusion_matrix'],index=['Thật 0','Thật 1'],columns=['Dự đoán 0','Dự đoán 1']))\n    st.subheader('Độ nhạy theo random seed trên cùng split')\n    st.dataframe(pd.read_csv(BASE/'seed_results.csv').set_index('seed'),use_container_width=True)\n    pr=pd.read_csv(BASE/'pr_curve_seed42.csv')\n    st.subheader('Precision–Recall trên test, seed 42')\n    st.line_chart(pr.set_index('recall')['precision'])\n    trade=pd.read_csv(BASE/'threshold_tradeoff.csv')\n    st.subheader('Đánh đổi giữa tỷ lệ cảnh báo và Recall')\n    st.line_chart(trade.pivot_table(index='alert_rate',columns='model',values='recall',aggfunc='max').sort_index())\n    st.warning('Nhãn sinh từ quy tắc rủi ro + nhiễu trên cùng bộ đặc trưng. Chỉ chứng minh pipeline trên dữ liệu tổng hợp; không dự báo hiệu quả tại MB và không suy ra giá trị tiền được bảo vệ. Tỷ lệ cảnh báo sai là tải kiểm tra tiềm năng, không phải giao dịch bị chặn.')\nwith architecture_tab:\n    st.subheader('Phần đã cài và phần đề xuất')\n    st.code('ĐÃ CÀI: giao diện Streamlit → kiểm tra 8 trường → XGBoost seed 42 → chính sách demo → lịch sử phiên/CSV\\nĐỀ XUẤT (chưa tích hợp): kênh giao dịch → kiểm soát bắt buộc → feature service → scoring → policy có người sở hữu → xử lý giao dịch / hàng đợi chuyên viên\\nCHƯA CÓ: Kafka, Core Banking, xác thực khách hàng, phân quyền, lưu trữ bền vững, HA/DR, quy trình khiếu nại, kiểm soát pháp lý production.')\n    st.write('Kênh và tín hiệu vị trí/thiết bị trong giao diện đều mô phỏng. Trước pilot phải xác minh khả năng thu thập đúng thời điểm, chất lượng nhãn và quyền xử lý dữ liệu; lỗi model hoặc thiếu dữ liệu không đồng nghĩa giao dịch an toàn.')\n    st.write('**Ranh giới quyết định:** PoC phát ra điểm và gợi ý thao tác cho người xem demo; không gửi lệnh chuyển tiền, giữ tiền, khóa tài khoản, OTP, FaceID hay thông báo cho khách hàng.')\n", "requirements.txt": "numpy>=1.26,<3\npandas>=2.2,<4\nscikit-learn>=1.4,<2\nxgboost>=2.1,<4\nstreamlit>=1.40,<2\npytest>=8,<10\n"}
for name, source in FILES.items():
    Path(name).write_text(source, encoding="utf-8")
print("Wrote", ", ".join(FILES))


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
!python train_model.py

import json, pandas as pd
from IPython.display import display
metrics = json.loads(open("metrics.json", encoding="utf-8").read())
display(pd.DataFrame(metrics["metrics"]["xgboost_by_seed"]).T)
display(pd.DataFrame(metrics["metrics"]["xgboost_seed_summary"]).T)


## Mở UI trong phiên Colab

Cell sau sử dụng cửa sổ proxy của Colab; không tạo tunnel URL công khai. Việc mở proxy cần được xác minh trong runtime/tài khoản dùng demo.


In [ ]:
import subprocess, time, urllib.request
from google.colab import output
streamlit_log = open("streamlit.log", "w")
streamlit_process = subprocess.Popen(["python", "-m", "streamlit", "run", "app.py", "--server.port=8501", "--server.headless=true"], stdout=streamlit_log, stderr=subprocess.STDOUT)
for _ in range(45):
    try:
        urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(open("streamlit.log").read())
output.serve_kernel_port_as_window(8501)
